# Recifind: Sistem Temu Kembali Informasi untuk Resep Masakan

Notebook ini menjalankan alur STKI yang sama dengan web Recifind: preprocessing, inverted index, pembobotan TF-IDF, cosine similarity, dan evaluasi.

- Data: `data/resep.csv` (26 resep, disusun sendiri)
- Hanya memakai pustaka standar Python, tanpa instalasi tambahan

## 1. Memuat dataset

In [1]:
import csv, io, math, os, re, urllib.request

URL = "https://raw.githubusercontent.com/Keidhnn/stki-resep-masakan/main/data/resep.csv"

def muat_teks():
    if os.path.exists("resep.csv"):                      # file lokal (hasil unggah atau unduhan)
        return open("resep.csv", encoding="utf-8-sig").read()
    try:                                                 # ambil dari GitHub
        return urllib.request.urlopen(URL).read().decode("utf-8-sig")
    except Exception:                                    # cadangan: unggah manual di Colab
        from google.colab import files
        unggah = files.upload()
        return list(unggah.values())[0].decode("utf-8-sig")

RESEP = list(csv.DictReader(io.StringIO(muat_teks())))
print("Jumlah resep:", len(RESEP))
print("Contoh:", RESEP[0]["nama"], "|", RESEP[0]["bahan"])

Jumlah resep: 26
Contoh: Ayam Goreng Bawang Putih | ayam;bawang putih;garam;merica;minyak goreng


## 2. Preprocessing

Huruf kecil, tokenisasi, lalu penghapusan stopword.

In [2]:
STOPWORD = set("dan di ke dari untuk dengan yang atau".split())

def tokenisasi(teks):
    kata = re.findall(r"[^\W_]+", teks.lower())          # huruf kecil + tokenisasi
    return [k for k in kata if k not in STOPWORD]        # hapus stopword

print(tokenisasi("Ayam goreng dengan bawang putih dan garam"))

['ayam', 'goreng', 'bawang', 'putih', 'garam']


## 3. Inverted index

Struktur: kata -> {nomor resep: frekuensi}. Teks yang diindeks adalah nama, kategori, dan bahan.

In [3]:
def teks_resep(r):
    return r["nama"] + " " + r["kategori"] + " " + r["bahan"].replace(";", " ")

token_dok = [tokenisasi(teks_resep(r)) for r in RESEP]

indeks = {}
for i, tokens in enumerate(token_dok):
    for t in tokens:
        indeks.setdefault(t, {})
        indeks[t][i] = indeks[t].get(i, 0) + 1

print("Kata unik:", len(indeks))
for k in ["ayam", "bawang", "tempe"]:
    print(k, "-> DF =", len(indeks[k]))

Kata unik: 101
ayam -> DF = 6
bawang -> DF = 19
tempe -> DF = 1


## 4. Pembobotan TF-IDF

IDF = log10(1 + N / DF). Bobot = TF x IDF.

In [4]:
N = len(RESEP)

def idf(t):
    return math.log10(1 + N / len(indeks[t]))

def vektor(tokens):
    v = {}
    for t in tokens:
        if t in indeks:
            v[t] = v.get(t, 0) + 1                       # TF
    return {t: tf * idf(t) for t, tf in v.items()}       # TF x IDF

v_dok = [vektor(tokens) for tokens in token_dok]
print("IDF ayam  :", round(idf("ayam"), 3))
print("IDF bawang:", round(idf("bawang"), 3))

IDF ayam  : 0.727
IDF bawang: 0.374


## 5. Cosine similarity dan ranking

In [5]:
def panjang(v):
    return math.sqrt(sum(x * x for x in v.values()))

def cari(query):
    vq = vektor(tokenisasi(query))
    pq = panjang(vq)
    hasil = []
    for i, vd in enumerate(v_dok):
        dot = sum(w * vd[t] for t, w in vq.items() if t in vd)
        pd = panjang(vd)
        skor = dot / (pq * pd) if pq and pd else 0
        if skor > 0:
            hasil.append((i, skor))
    return sorted(hasil, key=lambda x: -x[1])            # urut dari skor tertinggi

def tampil(query, n=5):
    print("Query:", query)
    hasil = cari(query)
    if not hasil:
        print("  (tidak ada hasil)")
    for i, skor in hasil[:n]:
        print(f"  {skor:.3f}  {RESEP[i]['nama']}")

tampil("ayam bawang putih")
tampil("telur cabai")
tampil("pizza")

Query: ayam bawang putih
  0.777  Ayam Goreng Bawang Putih
  0.567  Soto Ayam
  0.534  Sop Ayam
  0.523  Bubur Ayam
  0.423  Sate Ayam Bumbu Kacang
Query: telur cabai
  0.607  Telur Balado
  0.271  Mie Goreng Telur
  0.243  Nasi Goreng Kampung
  0.191  Gado-Gado
  0.155  Perkedel Kentang
Query: pizza
  (tidak ada hasil)


## 6. Evaluasi: Precision@5, Recall@5, dan F1@5

Daftar resep relevan di bawah ditetapkan sebelum pengujian (sama dengan kolom D pada `evaluasi-recifind.xlsx`). Query "pizza" tidak ikut dihitung rata-rata.

In [6]:
RELEVAN = {
    "ayam": ["Ayam Goreng Bawang Putih", "Soto Ayam", "Capcay Sayur", "Sate Ayam Bumbu Kacang", "Sop Ayam", "Bubur Ayam"],
    "telur": ["Nasi Goreng Kampung", "Soto Ayam", "Mie Goreng Telur", "Telur Balado", "Cumi Goreng Tepung", "Perkedel Kentang", "Gado-Gado"],
    "tempe": ["Tempe Orek"],
    "santan": ["Rendang Daging", "Es Cendol", "Kolak Pisang"],
    "pisang": ["Pisang Goreng", "Kolak Pisang"],
    "ayam bawang putih": ["Ayam Goreng Bawang Putih", "Soto Ayam", "Capcay Sayur", "Sate Ayam Bumbu Kacang", "Sop Ayam", "Bubur Ayam"],
    "telur cabai": ["Nasi Goreng Kampung", "Telur Balado", "Gado-Gado"],
    "kentang telur": ["Perkedel Kentang", "Gado-Gado"],
    "wortel kol": ["Capcay Sayur"],
    "gula merah santan": ["Es Cendol", "Kolak Pisang"],
    "soto": ["Soto Ayam"],
    "rendang": ["Rendang Daging"],
    "bakso": ["Bakso Sapi"],
    "bawang putih garam": ["Ayam Goreng Bawang Putih", "Bakso Sapi", "Perkedel Kentang"],
}

hasil_eval = []
print(f"{'Query':<22}{'R':>3}{'Tepat':>7}{'P@5':>7}{'R@5':>7}{'F1@5':>7}")
for q, rel in RELEVAN.items():
    top5 = [RESEP[i]["nama"] for i, _ in cari(q)[:5]]
    tepat = sum(1 for n in top5 if n in rel)
    p = tepat / 5
    r = tepat / len(rel)
    f1 = 2 * p * r / (p + r) if p + r else 0
    hasil_eval.append((p, r, f1))
    print(f"{q:<22}{len(rel):>3}{tepat:>7}{p:>7.2f}{r:>7.2f}{f1:>7.2f}")

n = len(hasil_eval)
print("-" * 51)
print(f"{'Rata-rata':<32}{sum(x[0] for x in hasil_eval)/n:>7.2f}{sum(x[1] for x in hasil_eval)/n:>7.2f}{sum(x[2] for x in hasil_eval)/n:>7.2f}")
print("Query pizza menghasilkan", len(cari("pizza")), "resep")

Query                   R  Tepat    P@5    R@5   F1@5
ayam                    6      5   1.00   0.83   0.91
telur                   7      5   1.00   0.71   0.83
tempe                   1      1   0.20   1.00   0.33
santan                  3      3   0.60   1.00   0.75
pisang                  2      2   0.40   1.00   0.57
ayam bawang putih       6      5   1.00   0.83   0.91
telur cabai             3      3   0.60   1.00   0.75
kentang telur           2      2   0.40   1.00   0.57
wortel kol              1      1   0.20   1.00   0.33
gula merah santan       2      2   0.40   1.00   0.57
soto                    1      1   0.20   1.00   0.33
rendang                 1      1   0.20   1.00   0.33
bakso                   1      1   0.20   1.00   0.33
bawang putih garam      3      3   0.60   1.00   0.75
---------------------------------------------------
Rata-rata                          0.50   0.96   0.59
Query pizza menghasilkan 0 resep


## Catatan

- Precision@5 selalu dibagi 5, padahal sebagian besar query hanya punya 1 sampai 3 resep relevan. Karena itu nilainya tidak bisa mencapai 1, walau semua resep relevan sudah muncul di urutan atas.
- Hasil notebook ini sama dengan hasil web Recifind karena memakai rumus dan data yang sama.